# Workshop — RAG sobre papers académicos con LangChain y NVIDIA NIM

En este workshop construirás un sistema RAG completo alimentado con dos papers fundamentales:

- **Attention Is All You Need** — Vaswani et al. (2017)
- **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks** — Lewis et al. (2020)

El notebook descarga los PDFs desde arXiv, extrae su texto, crea embeddings y responde preguntas usando únicamente el contenido indexado.

**Stack:** MarkItDown · LangChain · NVIDIA NIM · Qdrant

| Parte | Tarea |
|---|---|
| 0 | Instalar y configurar Colab |
| 1 | Descargar y convertir PDFs con MarkItDown |
| 2 | Crear `Document` y aplicar chunking |
| 3 | Generar embeddings NVIDIA e indexar en Qdrant |
| 4 | Construir el pipeline RAG con LCEL |
| 5 | Evaluar mediante preguntas con *ground truth* |

**Modelos:**
- Embeddings: `nvidia/nemotron-3-embed-1b`
- Generación: `nvidia/nemotron-3.5-super-vl-120b-a12b` (mi cuenta NVIDIA no tiene habilitado `nemotron-3-super-120b-a12b`, responde 404; este es del mismo tamaño, 120B con 12B activos)


In [1]:
# Dependencias para Google Colab
%pip install -qU "markitdown[pdf]" langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv requests



[notice] A new release of pip is available: 26.1.2 -> 26.2.2
[notice] To update, run: /Users/juandavidsierraorozco/.venvs/rag/bin/python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import getpass
import time
import random
from pathlib import Path
from typing import List

import requests
from dotenv import load_dotenv

# Busca un .env en la carpeta actual y en las dos carpetas padre
# (en este repo el .env está un nivel arriba de notebooks/).
for _d in [Path.cwd(), *list(Path.cwd().parents)[:2]]:
    if (_d / ".env").exists():
        load_dotenv(_d / ".env")
        break

NVIDIA_API_KEY = None
try:
    from google.colab import userdata          # Colab Secrets
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except Exception:
    NVIDIA_API_KEY = None
if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY   # la clave nunca se imprime

# CAMBIO DE MODELO: el original del taller es "nvidia/nemotron-3-super-120b-a12b",
# pero mi cuenta NVIDIA no lo tiene habilitado (responde 404).
# Uso "nvidia/nemotron-3.5-super-vl-120b-a12b": mismo tamaño (120B, 12B activos).
# Es un modelo con razonamiento, por eso subo max_completion_tokens más abajo.
GENERATION_MODEL = "nvidia/nemotron-3.5-super-vl-120b-a12b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"

print("✅ Configuración NVIDIA NIM lista")
print("   Generación:", GENERATION_MODEL)
print("   Embeddings:", EMBEDDING_MODEL)


✅ Configuración NVIDIA NIM lista
   Generación: nvidia/nemotron-3.5-super-vl-120b-a12b
   Embeddings: nvidia/nemotron-3-embed-1b


In [3]:
# Ayudas para compartir la clave NVIDIA sin ráfagas: reintentos con backoff ante 429/5xx
# y una pausa corta entre llamadas. Llamadas siempre secuenciales.
TRANSIENT = ("429", "500", "502", "503", "504", "timed out", "timeout", "Timeout",
             "rate limit", "Rate limit", "Too Many", "temporarily", "Connection", "empty response")
PAUSE = 0.7


def retry_call(fn, *args, tries: int = 7, base: float = 2.0, **kwargs):
    for attempt in range(tries):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            transient = any(t in str(exc) for t in TRANSIENT)
            if attempt == tries - 1 or not transient:
                raise
            wait = base * (2 ** attempt) + random.random()
            print(f"  ⏳ error transitorio ({type(exc).__name__}); reintento {attempt + 1} en {wait:.0f}s")
            time.sleep(wait)


def _invoke_nonempty(chain, question):
    out = chain.invoke(question)
    if not out or not out.strip():
        raise RuntimeError("empty response (el modelo gastó los tokens razonando)")
    return out


def ask(chain, question: str) -> str:
    """Invoca la cadena RAG con reintentos y pausa."""
    out = retry_call(_invoke_nonempty, chain, question)
    time.sleep(PAUSE)
    return out


def retrieve(retr, question: str):
    out = retry_call(retr.invoke, question)
    time.sleep(0.3)
    return out


def add_in_batches(store, docs, batch_size: int = 50):
    """Indexa por lotes para no saturar la API de embeddings."""
    ids = []
    for i in range(0, len(docs), batch_size):
        ids += retry_call(store.add_documents, docs[i:i + batch_size])
        time.sleep(0.5)
    return ids


---
## Parte 1 — Descargar y convertir los PDFs

La celda descarga los artículos únicamente si todavía no existen en `papers/`. Después, MarkItDown convierte cada PDF a texto conservando buena parte de su estructura.

> Si una sesión de Colab se reinicia, los archivos temporales se pierden y se descargarán nuevamente.


In [4]:
from markitdown import MarkItDown

# Usa papers/ si ya existe (en este repo está un nivel arriba de notebooks/);
# si no existe en ningún lado, la crea y descarga los PDFs (caso Colab).
DOCS_DIR = next((p for p in (Path("papers"), Path("../papers")) if p.exists()), Path("papers"))
DOCS_DIR.mkdir(parents=True, exist_ok=True)

PAPERS = {
    "attention_paper.pdf": "https://arxiv.org/pdf/1706.03762",
    "rag_paper.pdf": "https://arxiv.org/pdf/2005.11401",
}


def download_file(url: str, destination: Path) -> None:
    if destination.exists() and destination.stat().st_size > 10_000:
        print(f"↪ Ya existe: {destination.name}")
        return

    response = requests.get(url, timeout=90)
    response.raise_for_status()
    if not response.content.startswith(b"%PDF"):
        raise ValueError(f"La descarga de {url} no produjo un PDF válido")
    destination.write_bytes(response.content)
    print(f"⬇ Descargado: {destination.name} ({destination.stat().st_size / 1e6:.1f} MB)")


for filename, url in PAPERS.items():
    download_file(url, DOCS_DIR / filename)

md_converter = MarkItDown(enable_plugins=False)


def load_pdf(path: str | Path) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Archivo no encontrado: {path}")
    result = md_converter.convert(str(path))
    text = result.text_content
    if not text or not text.strip():
        raise ValueError(f"No se pudo extraer texto de {path.name}")
    return text


attention_text = load_pdf(DOCS_DIR / "attention_paper.pdf")
rag_text = load_pdf(DOCS_DIR / "rag_paper.pdf")

print(f"✅ attention_paper.pdf → {len(attention_text):,} caracteres")
print(f"✅ rag_paper.pdf       → {len(rag_text):,} caracteres")
print(f"\nInicio del paper de Attention:\n{attention_text[:300]}...")


↪ Ya existe: attention_paper.pdf
↪ Ya existe: rag_paper.pdf


✅ attention_paper.pdf → 40,174 caracteres
✅ rag_paper.pdf       → 69,121 caracteres

Inicio del paper de Attention:
Providedproperattributionisprovided,Googleherebygrantspermissionto
reproducethetablesandfiguresinthispapersolelyforuseinjournalisticor
scholarlyworks.
Attention Is All You Need
AshishVaswani∗ NoamShazeer∗ NikiParmar∗ JakobUszkoreit∗
GoogleBrain GoogleBrain GoogleResearch GoogleResearch
avaswani@goog...


---
## Parte 2 — Convertir a `Document` y aplicar chunking

LangChain utiliza objetos `Document` con dos campos principales:

- `page_content`: contenido textual del fragmento.
- `metadata`: título, archivo de origen e índice del fragmento.

Usaremos fragmentos de 900 caracteres con solapamiento de 120. Es un punto de partida didáctico; no es un valor universal.


In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


def pdf_to_documents(text: str, title: str, source: str) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=900,
        chunk_overlap=120,
        separators=["\n\n", "\n", ".", " "],
    )
    documents = []
    for index, chunk in enumerate(splitter.split_text(text)):
        if len(chunk.strip()) < 30:
            continue
        documents.append(Document(
            page_content=chunk,
            metadata={"title": title, "source": source, "chunk_index": index},
        ))
    return documents


attention_docs = pdf_to_documents(
    attention_text,
    title="Attention Is All You Need",
    source="attention_paper.pdf",
)
rag_docs = pdf_to_documents(
    rag_text,
    title="Retrieval-Augmented Generation",
    source="rag_paper.pdf",
)
all_docs = attention_docs + rag_docs

print(f"Attention → {len(attention_docs)} chunks")
print(f"RAG       → {len(rag_docs)} chunks")
print(f"Total     → {len(all_docs)} chunks")
print("\nEjemplo de metadatos:", all_docs[0].metadata)


Attention → 55 chunks
RAG       → 95 chunks
Total     → 150 chunks

Ejemplo de metadatos: {'title': 'Attention Is All You Need', 'source': 'attention_paper.pdf', 'chunk_index': 0}


---
## Parte 3 — Embeddings NVIDIA e índice Qdrant

`NVIDIAEmbeddings` distingue automáticamente entre embeddings de documentos y consultas mediante `embed_documents()` y `embed_query()`. La dimensión se obtiene directamente del modelo para evitar valores codificados manualmente.

Qdrant se ejecuta en memoria; al reiniciar el runtime se reconstruye el índice.


In [6]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

COLLECTION = "academic_papers_nvidia"

embeddings = NVIDIAEmbeddings(
    model=EMBEDDING_MODEL,
    truncate="END",
)
EMBEDDING_DIM = len(embeddings.embed_query("dimension probe"))

qdrant_client = QdrantClient(":memory:")
qdrant_client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
)

vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings,
)
ids = add_in_batches(vectorstore, all_docs)

print(f"✅ {len(ids)} chunks indexados en Qdrant")
print(f"✅ Dimensión de embeddings: {EMBEDDING_DIM}")


/Users/juandavidsierraorozco/.venvs/rag/lib/python3.12/site-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


✅ 150 chunks indexados en Qdrant
✅ Dimensión de embeddings: 2048


---
## Parte 4 — Pipeline RAG con LCEL

El pipeline conecta recuperación, construcción del prompt, generación con Nemotron y extracción del texto final. El prompt obliga al modelo a reconocer cuando el contexto recuperado no contiene la respuesta.


In [7]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatNVIDIA(
    model=GENERATION_MODEL,
    temperature=0.2,
    top_p=0.95,
    max_completion_tokens=1500,   # modelo con razonamiento: 800 podía quedarse corto
)

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 5, "score_threshold": 0.25},
)


def format_docs(documents: List[Document]) -> str:
    return "\n\n---\n\n".join(
        f"[{doc.metadata.get('title', 'Untitled')} | {doc.metadata.get('source', '')}]"
        f"\n{doc.page_content}"
        for doc in documents
    )


prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer using only the supplied academic-paper excerpts. "
        "If the excerpts do not contain enough evidence, say so explicitly. "
        "Answer in the same language as the question and be concise. "
        "Mention the paper title supporting the answer.",
    ),
    ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
])

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

test_question = "What is the Transformer architecture?"
test_answer = ask(rag_chain, test_question)
print("Pregunta:", test_question)
print("Respuesta:", test_answer)


Pregunta: What is the Transformer architecture?
Respuesta: The Transformer is a sequence transduction architecture introduced in “Attention Is All You Need” that relies entirely on attention mechanisms, dispensing with recurrence and convolutions. It uses an encoder–decoder structure built from stacked self-attention and point-wise fully connected layers. The encoder is a stack of N = 6 identical layers; each layer has multi-head self-attention followed by a position-wise feed-forward network, with residual connections and layer normalization around each sub-layer, and a model dimension of d = 512. This design allows significantly more parallelization and faster training than recurrent or convolutional encoder–decoder models.


---
## Parte 5 — Quiz sobre los papers

Ejecuta cada pregunta y compara la respuesta del RAG con el *ground truth*. Las preguntas están en inglés para coincidir con el idioma de los artículos; el prompt pide responder en el mismo idioma de la consulta.

Además de la corrección factual, observa si el contexto recuperado proviene del paper adecuado.


In [8]:
# ── Pregunta 1 ────────────────────────────────────────────────
q1 = "What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?"

answer1 = ask(rag_chain, q1)
print(f"Q: {q1}")
print(f"A: {answer1}")

Q: What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?
A: 28.4 BLEU. (Paper: "Attention Is All You Need")


<details>
<summary>✅ Ground Truth — Pregunta 1</summary>

El Transformer (big model) obtuvo un BLEU score de **28.4** en la tarea de
traducción WMT 2014 English-to-German, superando los mejores resultados
previos (incluyendo ensembles) por más de 2 puntos BLEU.
*(Fuente: Abstract y Section 6.1 — Attention Is All You Need)*
</details>

In [9]:
# ── Pregunta 2 ────────────────────────────────────────────────
q2 = "How does RAG combine parametric and non-parametric memory?"

answer2 = ask(rag_chain, q2)
print(f"Q: {q2}")
print(f"A: {answer2}")

  ⏳ error transitorio (Exception); reintento 1 en 2s


  ⏳ error transitorio (Exception); reintento 2 en 4s


Q: How does RAG combine parametric and non-parametric memory?
A: According to the paper **Retrieval-Augmented Generation** (rag_paper.pdf), RAG combines parametric and non-parametric memory by endowing a pre-trained parametric seq2seq model (e.g., BART) with a non-parametric dense vector index of Wikipedia accessed through a pre-trained neural retriever (DPR). The components are integrated in a probabilistic model trained end-to-end: the retriever provides latent documents conditioned on the input, and the seq2seq generator conditions on these retrieved documents together with the input to generate the output. Latent documents are marginalized with a top-K approximation, either on a per-output basis (same document for all tokens) or a per-token basis (different documents per token). Both the generator and retriever are jointly fine-tuned.


<details>
<summary>✅ Ground Truth — Pregunta 2</summary>

RAG combina **memoria paramétrica** (un modelo seq2seq pre-entrenado, BART-large)
con **memoria no-paramétrica** (un índice denso de vectores de Wikipedia).
El retriever (DPR) recupera los top-K documentos relevantes para la query,
y el generador (BART) los usa como contexto adicional para producir la respuesta.
Ambos componentes se entrenan end-to-end de forma conjunta.
*(Fuente: Abstract y Section 2 — RAG paper)*
</details>

In [10]:
# ── Pregunta 3 ────────────────────────────────────────────────
q3 = "What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?"

answer3 = ask(rag_chain, q3)
print(f"Q: {q3}")
print(f"A: {answer3}")

  ⏳ error transitorio (ReadTimeout); reintento 1 en 3s


Q: What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?
A: The scaling factor \(1/\sqrt{d_k}\) counteracts the growth of dot-product magnitudes for large \(d_k\). The paper explains that when query and key components are independent with mean 0 and variance 1, their dot product has mean 0 and variance \(d_k\), so large \(d_k\) pushes the softmax into regions with extremely small gradients. Scaling by \(\sqrt{d_k}\) (i.e., dividing by \(\sqrt{d_k}\)) mitigates this effect.

Paper: *Attention Is All You Need*


<details>
<summary>✅ Ground Truth — Pregunta 3</summary>

El factor de escala **1/√dk** se aplica para contrarrestar el efecto de que,
con valores grandes de dk, los productos punto crecen en magnitud y empujan
la función softmax hacia regiones con gradientes muy pequeños.
Esto haría el entrenamiento inestable. La división por √dk mantiene los
productos punto en un rango con gradientes útiles.
*(Fuente: Section 3.2.1 — Attention Is All You Need)*
</details>

In [11]:
# ── Pregunta 4 ────────────────────────────────────────────────
q4 = "What are the two RAG model variants proposed in the paper and how do they differ?"

answer4 = ask(rag_chain, q4)
print(f"Q: {q4}")
print(f"A: {answer4}")

Q: What are the two RAG model variants proposed in the paper and how do they differ?
A: The paper *Retrieval-Augmented Generation* proposes two variants: **RAG-Sequence** and **RAG-Token**. They differ in how they marginalize over retrieved latent documents: **RAG-Sequence** uses the same retrieved document to predict every target token (per-output basis, one document responsible for all tokens), whereas **RAG-Token** can predict each target token based on a different retrieved passage (per-token basis, different documents responsible for different tokens).


<details>
<summary>✅ Ground Truth — Pregunta 4</summary>

Los dos variantes son:
- **RAG-Sequence**: usa el **mismo documento recuperado** para generar toda
  la secuencia de salida. Trata el documento como una variable latente única.
- **RAG-Token**: puede usar un **documento diferente para cada token** generado,
  permitiendo combinar información de múltiples fuentes en una sola respuesta.

RAG-Token es más flexible pero más costoso computacionalmente.
*(Fuente: Section 2.1 — RAG paper)*
</details>

In [12]:
# ── Pregunta 5 ────────────────────────────────────────────────
q5 = "How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?"

answer5 = ask(rag_chain, q5)
print(f"Q: {q5}")
print(f"A: {answer5}")

  ⏳ error transitorio (Exception); reintento 1 en 3s


  ⏳ error transitorio (Exception); reintento 2 en 4s


Q: How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?
A: The base Transformer model uses **8 attention heads** (h = 8), with key and value dimensions per head of **d_k = d_v = 64** (since d_model = 512 and d_k = d_v = d_model/h = 64).

Source: *Attention Is All You Need* (attention_paper.pdf), Section 3.2.2 / Multi-head attention description.


<details>
<summary>✅ Ground Truth — Pregunta 5</summary>

El Transformer base usa **h = 8** attention heads en paralelo.
Para cada cabeza, las dimensiones son **dk = dv = dmodel/h = 512/8 = 64**.
El costo computacional total es similar al de single-head attention con
dimensionalidad completa, gracias a las dimensiones reducidas por cabeza.
*(Fuente: Section 3.2.2 — Attention Is All You Need)*
</details>

---
## Bonus — Pregunta entre ambos papers

Esta pregunta exige conectar información de los dos artículos. Además de generar la respuesta, inspeccionaremos los títulos y archivos de los fragmentos recuperados.


In [13]:
q_bonus = ("The RAG paper uses FAISS for fast retrieval with an HNSW approximation. "
           "The Attention paper proposes the Transformer architecture. "
           "What seq2seq model does RAG use as its generator component, "
           "and is it based on the Transformer?")

answer_bonus = ask(rag_chain, q_bonus)

# Mostrar también qué chunks recuperó
docs_retrieved = retrieve(retriever, q_bonus)
sources = [(d.metadata["title"], d.metadata["source"]) for d in docs_retrieved]

print(f"Q: {q_bonus}\n")
print(f"A: {answer_bonus}\n")
print(f"Fuentes recuperadas:")
for title, source in sources:
    print(f"  [{title}] — {source}")

Q: The RAG paper uses FAISS for fast retrieval with an HNSW approximation. The Attention paper proposes the Transformer architecture. What seq2seq model does RAG use as its generator component, and is it based on the Transformer?

A: RAG uses **BART** as its generator component. The RAG paper describes the parametric memory as a “pre-trained seq2seq transformer” and identifies that model as BART [32], so yes, it is Transformer-based.  
Source: *Retrieval-Augmented Generation* (rag_paper.pdf).

Fuentes recuperadas:
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Attention Is All You Need] — attention_paper.pdf


<details>
<summary>✅ Ground Truth — Bonus</summary>

RAG usa **BART-large** como generador (Section 2.3 del RAG paper).
BART es un modelo seq2seq pre-entrenado basado en la arquitectura **Transformer**
(referencia [58] en el RAG paper = el paper "Attention Is All You Need").
El RAG paper cita explícitamente: "BART-large, a pre-trained seq2seq transformer".

Esta pregunta demuestra el valor de RAG: el sistema conecta información de
dos documentos distintos para construir una respuesta completa.
</details>

---
## B1 · Evidencia: contexto recuperado y verificación contra el *ground truth*

Para cada pregunta imprimo los 5 fragmentos que llegaron al modelo (con su score de relevancia que reporta LangChain) y reviso con un criterio **estricto y automático** si la respuesta contiene los datos del *ground truth*. El criterio es el mismo que usaré en B2: la respuesta cuenta como acierto solo si contiene **todos** los datos clave de la lista (no cuenta que "suene bien").

| Pregunta | Datos que debe contener la respuesta |
|---|---|
| Q1 | `28.4` |
| Q2 | memoria no paramétrica + `BART` + Wikipedia / índice denso / DPR |
| Q3 | `softmax` + `gradient(s)` + dot products grandes (large / grow / magnitude) |
| Q4 | `RAG-Sequence` + `RAG-Token` + mismo documento + documento distinto por token |
| Q5 | 8 cabezas + `64` |
| Bonus | `BART` + `Transformer` |

In [14]:
import re

QUIZ = {"Q1": q1, "Q2": q2, "Q3": q3, "Q4": q4, "Q5": q5}

# Todos los patrones de la lista deben aparecer en la respuesta (sin distinguir mayúsculas).
GT_PATTERNS = {
    "Q1": [r"28\.4"],
    "Q2": [r"non-?\s?parametric|nonparametric", r"BART", r"Wikipedia|dense (vector )?(index|retriev)|DPR"],
    "Q3": [r"softmax", r"gradient", r"large|grow|magnitude|big"],
    "Q4": [r"RAG-Sequence", r"RAG-Token", r"same (retrieved )?document|single document|one document|same retrieved",
           r"different|each (target |generated )?token|per.token"],
    "Q5": [r"\b8\b|eight", r"\b64\b"],
    "Bonus": [r"BART", r"transformer"],
}


def is_hit(qid: str, answer: str) -> bool:
    return all(re.search(p, answer, flags=re.I) for p in GT_PATTERNS[qid])


EXPECTED_PAPER = {"Q1": "attention_paper.pdf", "Q2": "rag_paper.pdf", "Q3": "attention_paper.pdf",
                  "Q4": "rag_paper.pdf", "Q5": "attention_paper.pdf", "Bonus": "ambos"}
ANSWERS_B1 = {"Q1": answer1, "Q2": answer2, "Q3": answer3, "Q4": answer4, "Q5": answer5, "Bonus": answer_bonus}
QUESTIONS_B1 = {**QUIZ, "Bonus": q_bonus}

b1_rows = []
for qid, question in QUESTIONS_B1.items():
    scored = retry_call(vectorstore.similarity_search_with_relevance_scores, question, k=5, score_threshold=0.25)
    time.sleep(0.3)
    n_att = sum(d.metadata["source"] == "attention_paper.pdf" for d, _ in scored)
    n_rag = sum(d.metadata["source"] == "rag_paper.pdf" for d, _ in scored)
    print(f"\n=== {qid} · esperado: {EXPECTED_PAPER[qid]}")
    for d, sc in scored:
        print(f"  {sc:.3f} | {d.metadata['source']:<20} | chunk {d.metadata['chunk_index']:>3} | "
              f"{d.page_content[:70].replace(chr(10), ' ')}")
    exp = EXPECTED_PAPER[qid]
    if exp == "ambos":
        paper_ok = n_att > 0 and n_rag > 0
    else:
        paper_ok = scored[0][0].metadata["source"] == exp
    hit = is_hit(qid, ANSWERS_B1[qid])
    print(f"  → chunks: attention={n_att}, rag={n_rag} | top-1 del paper esperado: {paper_ok} | "
          f"respuesta con todos los datos del ground truth: {hit}")
    b1_rows.append((qid, n_att, n_rag, paper_ok, hit))

print("\nResumen B1 (criterio estricto):", sum(r[4] for r in b1_rows), "de", len(b1_rows))



=== Q1 · esperado: attention_paper.pdf
  0.855 | attention_paper.pdf  | chunk  28 | ls hurtsperplexity,asthemodellearnstobemoreunsure,butimprovesaccuracya
  0.829 | attention_paper.pdf  | chunk   1 | mechanism. We propose a new simple network architecture, the Transform
  0.790 | attention_paper.pdf  | chunk  26 | Table2: TheTransformerachievesbetterBLEUscoresthanpreviousstate-of-the
  0.758 | attention_paper.pdf  | chunk  40 | attention,replacingtherecurrentlayersmostcommonlyusedinencoder-decoder
  0.749 | attention_paper.pdf  | chunk  31 | Table3: VariationsontheTransformerarchitecture. Unlistedvaluesareident
  → chunks: attention=5, rag=0 | top-1 del paper esperado: True | respuesta con todos los datos del ground truth: True



=== Q2 · esperado: rag_paper.pdf
  0.796 | rag_paper.pdf        | chunk   1 | trainedmodelswithadifferentiableaccessmechanismtoexplicitnon-parametri
  0.773 | rag_paper.pdf        | chunk   8 | basis(wheredifferentdocumentsareresponsiblefordifferenttokens). LikeT5
  0.760 | rag_paper.pdf        | chunk  32 | findevidenceforthishypothesisbyfeedingtheBART-onlybaselinewiththeparti
  0.753 | rag_paper.pdf        | chunk   7 | Weendowpre-trained,parametric-memorygenerationmodelswithanon-parametri
  0.741 | rag_paper.pdf        | chunk  50 | toprovideafinaloutput. Theseapproacheshaveprovedsuccessfulinanumberofd
  → chunks: attention=0, rag=5 | top-1 del paper esperado: True | respuesta con todos los datos del ground truth: True



=== Q3 · esperado: attention_paper.pdf
  0.775 | attention_paper.pdf  | chunk  11 | ScaledDot-ProductAttention Multi-HeadAttention Figure 2: (left) Scaled
  0.741 | attention_paper.pdf  | chunk  12 | thematrixofoutputsas: QKT Attention(Q,K,V)=softmax( √ )V (1) d k Thetw
  0.690 | attention_paper.pdf  | chunk  13 | k extremelysmallgradients4. Tocounteractthiseffect,wescalethedotproduc
  0.648 | attention_paper.pdf  | chunk  16 | allpositionsinthedecoderuptoandincludingthatposition. Weneedtopreventl
  0.642 | attention_paper.pdf  | chunk  20 | versionsproducednearlyidenticalresults(seeTable3row(E)).Wechosethesinu
  → chunks: attention=5, rag=0 | top-1 del paper esperado: True | respuesta con todos los datos del ground truth: True



=== Q4 · esperado: rag_paper.pdf
  0.755 | rag_paper.pdf        | chunk  10 | 2 Methods WeexploreRAGmodels,whichusetheinputsequencextoretrievetextdo
  0.737 | rag_paper.pdf        | chunk  11 | byθthatgeneratesacurrenttokenbasedonacontextofthepreviousi−1tokensy ,t
  0.736 | rag_paper.pdf        | chunk   1 | trainedmodelswithadifferentiableaccessmechanismtoexplicitnon-parametri
  0.732 | rag_paper.pdf        | chunk  50 | toprovideafinaloutput. Theseapproacheshaveprovedsuccessfulinanumberofd
  0.725 | rag_paper.pdf        | chunk   7 | Weendowpre-trained,parametric-memorygenerationmodelswithanon-parametri
  → chunks: attention=0, rag=5 | top-1 del paper esperado: True | respuesta con todos los datos del ground truth: True



=== Q5 · esperado: attention_paper.pdf
  0.774 | attention_paper.pdf  | chunk  14 | output values. These are concatenated and once again projected, result
  0.744 | attention_paper.pdf  | chunk  15 | 3.2.3 ApplicationsofAttentioninourModel TheTransformerusesmulti-headat
  0.744 | attention_paper.pdf  | chunk   9 | Figure1: TheTransformer-modelarchitecture. TheTransformerfollowsthisov
  0.736 | attention_paper.pdf  | chunk   0 | Providedproperattributionisprovided,Googleherebygrantspermissionto rep
  0.728 | attention_paper.pdf  | chunk  40 | attention,replacingtherecurrentlayersmostcommonlyusedinencoder-decoder
  → chunks: attention=5, rag=0 | top-1 del paper esperado: True | respuesta con todos los datos del ground truth: True



=== Bonus · esperado: ambos
  0.765 | rag_paper.pdf        | chunk   7 | Weendowpre-trained,parametric-memorygenerationmodelswithanon-parametri
  0.756 | rag_paper.pdf        | chunk  11 | byθthatgeneratesacurrenttokenbasedonacontextofthepreviousi−1tokensy ,t
  0.756 | rag_paper.pdf        | chunk  10 | 2 Methods WeexploreRAGmodels,whichusetheinputsequencextoretrievetextdo
  0.737 | rag_paper.pdf        | chunk  12 | ismarginalizedtogettheseq2seqprobabilityp(y|x)viaatop-Kapproximation. 
  0.737 | attention_paper.pdf  | chunk   0 | Providedproperattributionisprovided,Googleherebygrantspermissionto rep
  → chunks: attention=1, rag=4 | top-1 del paper esperado: True | respuesta con todos los datos del ground truth: True

Resumen B1 (criterio estricto): 6 de 6


## B1 · Ejecuta el RAG de papers y evalúa con el ground truth

_Evidencia:_ notebook `notebooks/03_rag_workshop_papers_nvidia_nim_colab.ipynb` ejecutado de principio a fin (celda "B1 · Evidencia"). Índice de 150 chunks (dimensión de embeddings 2048). Generación con `nvidia/nemotron-3.5-super-vl-120b-a12b` porque mi cuenta no tiene habilitado `nemotron-3-super-120b-a12b` (404).

| # | Pregunta (resumen) | Respuesta del RAG (resumen) | Correcta | Paper correcto |
|---|---|---|---|---|
| Q1 | BLEU del Transformer big en WMT14 En→De | 28.4 BLEU, citando Attention Is All You Need | sí | sí (5 de 5 chunks de Attention, top-1 score 0.855) |
| Q2 | Cómo combina RAG memoria paramétrica y no paramétrica | Seq2seq preentrenado (BART) + índice denso de Wikipedia con retriever DPR, entrenados juntos | sí | sí (5 de 5 chunks de RAG) |
| Q3 | Para qué sirve el factor sqrt(dk) | Evita que los productos punto crezcan con dk grande y empujen el softmax a gradientes muy pequeños | sí | sí (5 de 5 de Attention; el chunk 13 trae "extremely small gradients") |
| Q4 | Las dos variantes de RAG | RAG-Sequence (mismo documento para toda la secuencia) y RAG-Token (documento distinto por token) | sí | sí (5 de 5 de RAG) |
| Q5 | Cabezas de atención del Transformer base y dims | 8 cabezas, dk = dv = 64 (512/8) | sí | sí (5 de 5 de Attention) |
| Bonus | Generador de RAG y si es Transformer | BART, y sí está basado en el Transformer | sí | parcial: aparecieron los dos papers (4 de RAG y 1 de Attention) |

Bonus, fragmentos recuperados: `[Retrieval-Augmented Generation] rag_paper.pdf` x4 y `[Attention Is All You Need] attention_paper.pdf` x1.

_Mi respuesta / análisis:_ Las 6 respuestas coinciden con el ground truth (6 de 6 también con mi comprobación automática estricta de palabras clave). En las cinco preguntas del quiz el contexto vino del paper correcto, con el top-1 del paper esperado en todas. En la bonus aparecieron los dos papers, pero el chunk de Attention que llegó fue el chunk 0 (la página de título y permisos), no la sección de arquitectura: la respuesta "BART es un transformer" salió del propio texto del paper de RAG ("pre-trained seq2seq transformer"), no de una conexión real entre los dos documentos. Por eso marco el paper como parcial en la bonus. Un detalle técnico: el texto que extrae MarkItDown del paper de Attention viene con muchas palabras pegadas ("Table2:TheTransformerachieves..."), y aun así la recuperación funcionó.


---
## B2 · Experimentos con `chunk_size`, `k` y `score_threshold`

Cambio un parámetro a la vez. Primero las funciones auxiliares de la guía (con dos cambios mínimos: indexado por lotes y reintentos), luego los 3 experimentos.

**Criterio de acierto (estricto y constante en las 9 configuraciones):** una pregunta cuenta como acierto si la respuesta contiene **todos** los patrones de `GT_PATTERNS` (la tabla de B1). Si el modelo dice que no tiene evidencia, o responde sin el dato exacto, no cuenta. Es un criterio de palabras clave: no mide estilo, solo si el dato del *ground truth* llegó a la respuesta.

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CORPUS = [
    (attention_text, "Attention Is All You Need", "attention_paper.pdf"),
    (rag_text, "Retrieval-Augmented Generation", "rag_paper.pdf"),
]


def build_store(chunk_size: int, chunk_overlap: int):
    """Chunking + embeddings + Qdrant en memoria. Devuelve (store, n_chunks)."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " "],
    )
    docs = []
    for text, title, source in CORPUS:
        for i, chunk in enumerate(splitter.split_text(text)):
            if len(chunk.strip()) >= 30:
                docs.append(Document(page_content=chunk,
                                     metadata={"title": title, "source": source, "chunk_index": i}))
    name = f"exp_{chunk_size}_{chunk_overlap}"
    client = QdrantClient(":memory:")
    client.create_collection(collection_name=name,
                             vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE))
    store = QdrantVectorStore(client=client, collection_name=name, embedding=embeddings)
    add_in_batches(store, docs)          # cambio mío: por lotes y con reintentos
    return store, len(docs)


def build_chain(store, k: int = 5, threshold: float = 0.25):
    """Reutiliza un índice ya construido y cambia solo la recuperación."""
    retr = store.as_retriever(search_type="similarity_score_threshold",
                              search_kwargs={"k": k, "score_threshold": threshold})
    chain = ({"context": retr | format_docs, "question": RunnablePassthrough()}
             | prompt | llm | StrOutputParser())
    return chain, retr


RESULTS = {}      # etiqueta -> dict con chunks del índice, recuperados promedio, aciertos y respuestas


def run_config(label: str, store, n_index: int, k: int, threshold: float):
    chain_, retr_ = build_chain(store, k=k, threshold=threshold)
    hits, retrieved, answers = [], [], {}
    for qid, question in QUIZ.items():
        retrieved.append(len(retrieve(retr_, question)))
        ans = ask(chain_, question)
        answers[qid] = ans
        hits.append(is_hit(qid, ans))
    RESULTS[label] = {"n_index": n_index, "k": k, "th": threshold, "retrieved": retrieved,
                      "avg_ret": sum(retrieved) / len(retrieved), "hits": hits,
                      "n_hits": sum(hits), "answers": answers}
    r = RESULTS[label]
    print(f"{label:<28} índice={n_index:>3} chunks | recuperados por pregunta={retrieved} "
          f"(prom {r['avg_ret']:.1f}) | aciertos={r['n_hits']}/5 {['✔' if h else '✘' for h in hits]}")
    return r


### Experimento 1 · `chunk_size` = 400, 900, 1500 (overlap ≈ 13 %, `k=5`, `score_threshold=0.25`)

Son 3 construcciones del índice (3 pasadas de embeddings sobre el corpus).

In [16]:
STORES = {}
for cs, ov in [(400, 52), (900, 120), (1500, 200)]:
    t0 = time.time()
    STORES[cs] = build_store(cs, ov)
    print(f"[chunk_size={cs}, overlap={ov}] {STORES[cs][1]} chunks indexados en {time.time() - t0:.0f}s")
    run_config(f"E1 chunk={cs}", STORES[cs][0], STORES[cs][1], k=5, threshold=0.25)


[chunk_size=400, overlap=52] 326 chunks indexados en 12s


  ⏳ error transitorio (ReadTimeout); reintento 1 en 2s


  ⏳ error transitorio (Exception); reintento 1 en 3s


E1 chunk=400                 índice=326 chunks | recuperados por pregunta=[5, 5, 5, 5, 5] (prom 5.0) | aciertos=4/5 ['✔', '✘', '✔', '✔', '✔']


[chunk_size=900, overlap=120] 150 chunks indexados en 6s


  ⏳ error transitorio (Exception); reintento 1 en 2s


E1 chunk=900                 índice=150 chunks | recuperados por pregunta=[5, 5, 5, 5, 5] (prom 5.0) | aciertos=5/5 ['✔', '✔', '✔', '✔', '✔']


[chunk_size=1500, overlap=200] 96 chunks indexados en 4s


E1 chunk=1500                índice= 96 chunks | recuperados por pregunta=[5, 5, 5, 5, 5] (prom 5.0) | aciertos=4/5 ['✔', '✔', '✔', '✘', '✔']


### Experimento 2 · `k` = 3, 5, 8 (reutilizo el índice de 900, `score_threshold=0.25`)

In [17]:
store_900, n900 = STORES[900]
for k in (3, 5, 8):
    run_config(f"E2 k={k}", store_900, n900, k=k, threshold=0.25)


E2 k=3                       índice=150 chunks | recuperados por pregunta=[3, 3, 3, 3, 3] (prom 3.0) | aciertos=5/5 ['✔', '✔', '✔', '✔', '✔']


E2 k=5                       índice=150 chunks | recuperados por pregunta=[5, 5, 5, 5, 5] (prom 5.0) | aciertos=5/5 ['✔', '✔', '✔', '✔', '✔']


E2 k=8                       índice=150 chunks | recuperados por pregunta=[8, 8, 8, 8, 8] (prom 8.0) | aciertos=5/5 ['✔', '✔', '✔', '✔', '✔']


### Experimento 3 · `score_threshold` = 0.15, 0.25, 0.40 (índice de 900, `k=5`)

In [18]:
for th in (0.15, 0.25, 0.40):
    run_config(f"E3 threshold={th:.2f}", store_900, n900, k=5, threshold=th)


E3 threshold=0.15            índice=150 chunks | recuperados por pregunta=[5, 5, 5, 5, 5] (prom 5.0) | aciertos=5/5 ['✔', '✔', '✔', '✔', '✔']


E3 threshold=0.25            índice=150 chunks | recuperados por pregunta=[5, 5, 5, 5, 5] (prom 5.0) | aciertos=5/5 ['✔', '✔', '✔', '✔', '✔']


E3 threshold=0.40            índice=150 chunks | recuperados por pregunta=[5, 5, 5, 5, 5] (prom 5.0) | aciertos=5/5 ['✔', '✔', '✔', '✔', '✔']


In [19]:
print(f"{'Configuración':<26}{'Índice':>7}{'Recup. prom.':>14}{'Aciertos':>10}   por pregunta Q1..Q5")
for label, r in RESULTS.items():
    print(f"{label:<26}{r['n_index']:>7}{r['avg_ret']:>14.1f}{str(r['n_hits']) + '/5':>10}   "
          + " ".join('✔' if h else '✘' for h in r['hits']))

print("\nRespuestas completas por configuración (para revisar a mano los falsos positivos/negativos):")
for label, r in RESULTS.items():
    print(f"\n##### {label}")
    for qid, a in r["answers"].items():
        print(f"[{qid}] {'✔' if is_hit(qid, a) else '✘'} {a.strip().replace(chr(10), ' ')[:420]}")


Configuración              Índice  Recup. prom.  Aciertos   por pregunta Q1..Q5
E1 chunk=400                  326           5.0       4/5   ✔ ✘ ✔ ✔ ✔
E1 chunk=900                  150           5.0       5/5   ✔ ✔ ✔ ✔ ✔
E1 chunk=1500                  96           5.0       4/5   ✔ ✔ ✔ ✘ ✔
E2 k=3                        150           3.0       5/5   ✔ ✔ ✔ ✔ ✔
E2 k=5                        150           5.0       5/5   ✔ ✔ ✔ ✔ ✔
E2 k=8                        150           8.0       5/5   ✔ ✔ ✔ ✔ ✔
E3 threshold=0.15             150           5.0       5/5   ✔ ✔ ✔ ✔ ✔
E3 threshold=0.25             150           5.0       5/5   ✔ ✔ ✔ ✔ ✔
E3 threshold=0.40             150           5.0       5/5   ✔ ✔ ✔ ✔ ✔

Respuestas completas por configuración (para revisar a mano los falsos positivos/negativos):

##### E1 chunk=400
[Q1] ✔ 28.4 BLEU. (Paper: "Attention Is All You Need")
[Q2] ✘ RAG combines parametric and non-parametric memory by endowing a pre-trained seq2seq transformer (the parametric me

### Diagnóstico extra: ¿en qué escala está el `score_threshold`?

En los tres umbrales de arriba noté que el número de chunks recuperados no cambia. Reviso los scores crudos para entender por qué: la guía no lo menciona, pero `langchain-qdrant` convierte el coseno `c` a un *score de relevancia* `(c + 1) / 2` antes de comparar con el umbral.

In [20]:
def relevance_scores(store, question, k=8):
    out = retry_call(store.similarity_search_with_relevance_scores, question, k=k)
    time.sleep(0.3)
    return [sc for _, sc in out]


print("Score de relevancia (coseno+1)/2 de los 8 mejores chunks por pregunta (índice de 900):")
TOP = {}
for qid, question in QUIZ.items():
    TOP[qid] = relevance_scores(store_900, question)
    print(f"  {qid}: " + " ".join(f"{s:.3f}" for s in TOP[qid]))

all_scores = [s for v in TOP.values() for s in v]
print(f"\nMínimo entre los 40 scores: {min(all_scores):.3f} (coseno {2 * min(all_scores) - 1:.3f})")
print("Umbrales del experimento 3 en coseno equivalente: "
      + ", ".join(f"{t:.2f} → {2 * t - 1:.2f}" for t in (0.15, 0.25, 0.40)))

print("\nChunks que sobreviven (k=8) a umbrales que sí actúan sobre esta escala:")
for th in (0.70, 0.75, 0.80, 0.85):
    counts = [sum(s >= th for s in v) for v in TOP.values()]
    print(f"  umbral {th:.2f}: {counts} (prom {sum(counts) / 5:.1f})")


Score de relevancia (coseno+1)/2 de los 8 mejores chunks por pregunta (índice de 900):


  Q1: 0.855 0.829 0.790 0.758 0.749 0.741 0.728 0.722


  Q2: 0.796 0.773 0.760 0.753 0.741 0.727 0.701 0.698


  Q3: 0.775 0.741 0.690 0.648 0.642 0.637 0.637 0.634


  Q4: 0.755 0.737 0.736 0.732 0.725 0.713 0.712 0.709


  Q5: 0.774 0.744 0.744 0.736 0.728 0.725 0.723 0.712

Mínimo entre los 40 scores: 0.634 (coseno 0.269)
Umbrales del experimento 3 en coseno equivalente: 0.15 → -0.70, 0.25 → -0.50, 0.40 → -0.20

Chunks que sobreviven (k=8) a umbrales que sí actúan sobre esta escala:
  umbral 0.70: [8, 7, 2, 8, 8] (prom 6.6)
  umbral 0.75: [4, 4, 1, 1, 1] (prom 2.2)
  umbral 0.80: [2, 0, 0, 0, 0] (prom 0.4)
  umbral 0.85: [1, 0, 0, 0, 0] (prom 0.2)


In [21]:
# Una configuración extra (fuera de las 9 pedidas) con un umbral que sí filtra en esta escala.
run_config("EXTRA threshold=0.80", store_900, n900, k=5, threshold=0.80)


No relevant docs were retrieved using the relevance score threshold 0.8


No relevant docs were retrieved using the relevance score threshold 0.8


No relevant docs were retrieved using the relevance score threshold 0.8


No relevant docs were retrieved using the relevance score threshold 0.8


  ⏳ error transitorio (Exception); reintento 1 en 3s


...

No relevant docs were retrieved using the relevance score threshold 0.8


No relevant docs were retrieved using the relevance score threshold 0.8


No relevant docs were retrieved using the relevance score threshold 0.8


EXTRA threshold=0.80         índice=150 chunks | recuperados por pregunta=[2, 0, 0, 0, 0] (prom 0.4) | aciertos=1/5 ['✔', '✘', '✘', '✘', '✘']


{'n_index': 150,
 'k': 5,
 'th': 0.8,
 'retrieved': [2, 0, 0, 0, 0],
 'avg_ret': 0.4,
 'hits': [True, False, False, False, False],
 'n_hits': 1,
 'answers': {'Q1': '28.4 BLEU. (Paper: "Attention Is All You Need")',
  'Q2': 'The supplied context contains no academic-paper excerpts, so there is insufficient evidence to answer how RAG combines parametric and non-parametric memory. No paper title can be cited.',
  'Q3': 'The supplied context contains no academic-paper excerpts, so there is insufficient evidence to answer from the provided material, and no supporting paper title can be cited from it.',
  'Q4': 'The supplied context is empty, so there is no evidence to identify any RAG model variants or the paper title. Please provide the relevant excerpts.',
  'Q5': 'The supplied context contains no academic-paper excerpts, so there is no evidence to determine the number of attention heads or the key/value dimensions per head for the base Transformer model. No paper title can be cited.'}}

## B2 · Experimentos con chunk_size, k y score_threshold

_Evidencia:_ celda de funciones auxiliares y celdas de los 3 experimentos en el notebook (3 construcciones de índice: 326, 150 y 96 chunks). Criterio de acierto, igual en las 9 configuraciones: la respuesta debe contener todos los datos clave del ground truth por palabras clave (Q1 "28.4"; Q2 no paramétrica + BART + Wikipedia/DPR; Q3 softmax + gradient + magnitud grande; Q4 RAG-Sequence + RAG-Token + mismo documento + documento distinto por token; Q5 8 y 64). Si el modelo no nombra el dato exacto, no cuenta.

| Experimento | Configuración | Chunks en el índice | Recuperados por pregunta | Aciertos | Observación |
|---|---|---|---|---|---|
| 1 chunk_size | 400 (overlap 52), k=5, th=0.25 | 326 | 5 | 4/5 | Falló Q2 por mi criterio: explica las dos memorias pero dice "pre-trained seq2seq transformer" sin nombrar BART. |
| 1 chunk_size | 900 (overlap 120), k=5, th=0.25 | 150 | 5 | 5/5 | Configuración del notebook. |
| 1 chunk_size | 1500 (overlap 200), k=5, th=0.25 | 96 | 5 | 4/5 | Falló Q4 por mi patrón, pero leyendo la respuesta es correcta (escribe "*same* retrieved document" con asteriscos y mi regex no lo detectó): falso negativo del criterio. |
| 2 k | 900, k=3 | 150 | 3 | 5/5 | Con 3 chunks alcanza para las 5 preguntas. |
| 2 k | 900, k=5 | 150 | 5 | 5/5 | Igual a la base. |
| 2 k | 900, k=8 | 150 | 8 | 5/5 | Más contexto sin ganar aciertos. |
| 3 threshold | 900, k=5, th=0.15 | 150 | 5 | 5/5 | Idéntico a la base. |
| 3 threshold | 900, k=5, th=0.25 | 150 | 5 | 5/5 | Base. |
| 3 threshold | 900, k=5, th=0.40 | 150 | 5 | 5/5 | Idéntico: el umbral no filtró nada. |

Diagnóstico extra (fuera de las 9): `langchain-qdrant` convierte el coseno c en un score de relevancia (c+1)/2 antes de comparar con el umbral. En mis 40 mejores resultados el score mínimo fue 0.634 (coseno 0.269), así que 0.15, 0.25 y 0.40 equivalen a cosenos de -0.70, -0.50 y -0.20 y nunca filtran. Con umbral 0.80 sí actúa: recuperó [2, 0, 0, 0, 0] chunks para Q1..Q5 y acertó 1/5 (en las demás, el modelo dijo que no tenía evidencia).

_Mi respuesta / análisis:_ El parámetro que más movió los resultados fue chunk_size: es el único que cambió el contenido de lo recuperado, y tanto 400 como 1500 sacaron 4/5 frente a 5/5 con 900. Con k (3, 5, 8) todo dio 5/5 y solo cambió cuánto contexto se paga; k=3 ya alcanzaba. El score_threshold de 0.15 a 0.40 no tuvo ningún efecto porque, con esta escala (c+1)/2 y este embedder, todos los chunks tienen score entre 0.63 y 0.86; un umbral útil estaría cerca de 0.75 a 0.80, y con 0.80 ya se pierden respuestas buenas (1/5). Debo ser prudente con la diferencia 4/5 contra 5/5: es de una sola pregunta, una fue un falso negativo de mi criterio, y no repetí 400 y 1500 para medir el ruido del modelo (la configuración 900/k5/0.25 corrió tres veces, una por experimento, y las tres dio 5/5). No encontré una configuración claramente mejor que la del notebook; en costo, k=3 con chunk 900 da lo mismo con menos contexto, y 900 es un buen punto medio entre los 326 chunks de 400 (más ruido por pieza) y los 96 de 1500.


---
## B3 · Preguntas trampa: ¿el RAG admite que no sabe?

Tres preguntas plausibles cuya respuesta **no está** en los dos papers (modelos y benchmarks posteriores a 2020 o ausentes). Primero compruebo con una búsqueda de texto que los términos clave de verdad no aparecen en el corpus; luego miro qué fragmentos recupera el retriever (si recupera algo con score alto, el modelo tiene "material" para inventar).

In [22]:
TRAPS = {
    "T1": "What BLEU score did GPT-4 achieve on WMT 2014 English-to-German?",
    "T2": "What exact-match score did RAG obtain on the HotpotQA multi-hop question answering benchmark?",
    "T3": "How many GPUs and how many training days did Llama 2 70B need, and how does that compare with the base Transformer?",
}
full_corpus = (attention_text + "\n" + rag_text).lower()
for term in ("gpt-4", "hotpotqa", "llama", "p100"):
    print(f"¿'{term}' aparece en el corpus? {term in full_corpus}")

PROMPT_V1_SYSTEM = prompt.messages[0].prompt.template
print("\nPROMPT v1 (el del notebook):\n", PROMPT_V1_SYSTEM)

def run_traps(chain_, label):
    out = {}
    for tid, question in TRAPS.items():
        out[tid] = ask(chain_, question)
        print(f"\n[{label}] {tid}: {question}\n→ {out[tid].strip()}")
    return out

print("\n--- Recuperación para cada trampa (score de relevancia) ---")
for tid, question in TRAPS.items():
    scored = retry_call(vectorstore.similarity_search_with_relevance_scores, question, k=5, score_threshold=0.25)
    time.sleep(0.3)
    print(f"{tid}: " + " | ".join(f"{sc:.3f} {d.metadata['source'][:4]}#{d.metadata['chunk_index']}" for d, sc in scored))


¿'gpt-4' aparece en el corpus? False
¿'hotpotqa' aparece en el corpus? False
¿'llama' aparece en el corpus? False
¿'p100' aparece en el corpus? True

PROMPT v1 (el del notebook):
 Answer using only the supplied academic-paper excerpts. If the excerpts do not contain enough evidence, say so explicitly. Answer in the same language as the question and be concise. Mention the paper title supporting the answer.

--- Recuperación para cada trampa (score de relevancia) ---


T1: 0.771 atte#28 | 0.754 atte#1 | 0.730 atte#26 | 0.701 atte#24 | 0.683 atte#40


T2: 0.736 rag_#29 | 0.726 rag_#25 | 0.710 rag_#35 | 0.708 rag_#20 | 0.702 rag_#42


T3: 0.702 atte#6 | 0.680 atte#29 | 0.677 atte#1 | 0.665 atte#28 | 0.663 atte#2


In [23]:
# Respuestas con el prompt v1 (el original)
traps_v1 = run_traps(rag_chain, "prompt v1")



[prompt v1] T1: What BLEU score did GPT-4 achieve on WMT 2014 English-to-German?
→ The provided excerpts from *Attention Is All You Need* do not mention GPT-4 or any BLEU score achieved by GPT-4 on WMT 2014 English-to-German. They only report Transformer results (e.g., 28.4 BLEU for English-to-German).



[prompt v1] T2: What exact-match score did RAG obtain on the HotpotQA multi-hop question answering benchmark?
→ The provided excerpts do not mention HotpotQA or report an exact-match score for it. The paper title is *Retrieval-Augmented Generation* (rag_paper.pdf).


  ⏳ error transitorio (ReadTimeout); reintento 1 en 3s


  ⏳ error transitorio (ReadTimeout); reintento 2 en 4s


  ⏳ error transitorio (Exception); reintento 3 en 8s



[prompt v1] T3: How many GPUs and how many training days did Llama 2 70B need, and how does that compare with the base Transformer?
→ The supplied excerpts do not contain any information about Llama 2 70B, so its GPU count and training days cannot be determined from this context. The excerpts are from “Attention Is All You Need,” which reports that the Transformer (big) was trained for 3.5 days on 8 P100 GPUs for WMT 2014 English-to-German (and the English-to-French big model for 3.5 days on eight GPUs); no Llama 2 70B comparison is possible from these excerpts.


## B3 · Preguntas trampa: ¿el RAG admite que no sabe?

_Evidencia:_ celdas de B3 del notebook. Primero verifiqué por búsqueda de texto que "gpt-4", "hotpotqa" y "llama" no aparecen en el corpus (los tres False).

| # | Pregunta trampa | Chunks recuperados (score) | Clasificación con prompt v1 | Con prompt v2 |
|---|---|---|---|---|
| T1 | ¿Qué BLEU logró GPT-4 en WMT14 En→De? | 5 de Attention, 0.771 a 0.683 | admite que no sabe (dice que no menciona GPT-4 y aclara que solo hay resultados del Transformer, 28.4) | admite que no sabe |
| T2 | ¿Qué exact-match obtuvo RAG en HotpotQA? | 5 de RAG, 0.736 a 0.702 | admite que no sabe | admite que no sabe |
| T3 | ¿Cuántas GPUs y días necesitó Llama 2 70B y cómo se compara con el Transformer base? | 5 de Attention, 0.702 a 0.663 | admite que no sabe de Llama 2 y agrega, separado, el dato real del Transformer (3.5 días en 8 P100) | admite que no sabe, mismo dato adicional |

Prompt de sistema antes (v1, el del notebook): "Answer using only the supplied academic-paper excerpts. If the excerpts do not contain enough evidence, say so explicitly. Answer in the same language as the question and be concise. Mention the paper title supporting the answer."

Prompt después (v2), diff: la primera frase pasa a "ONLY"; la segunda se reemplaza por "First check whether the excerpts explicitly contain the exact fact the question asks for (the same model, benchmark and metric). If they do not, say clearly that the supplied papers do not contain that information. You may add one short sentence about what related information the excerpts do contain, but never use outside knowledge, never guess numbers, and never transfer a result from one model or benchmark to a different one."; y la última añade "when there is evidence".

Regresión con v2: Q1 y Q3 siguieron acertando (la ejecución guardada tiene también Q5).

_Mi respuesta / análisis:_ Con el prompt original ninguna de las tres inventó: las tres admitieron que los excerpts no contienen el dato. Esto ocurrió aunque el retriever sí entregó 5 chunks por pregunta con scores de 0.66 a 0.77, es decir, "material" irrelevante pero por encima del umbral, que es justo el riesgo que advierte la pista; el modelo no lo usó para inventar. El caso más cercano a una mezcla fue T3, donde el modelo, además de reconocer que no hay nada de Llama 2, agregó el dato real del Transformer (3.5 días en 8 P100); lo clasifico como admite, porque no lo atribuyó a Llama 2, pero es el tipo de respuesta que un lector apurado podría confundir. Como nada inventó, el cambio de prompt no era necesario; lo probé de todos modos como medida preventiva y v2 mantuvo el comportamiento correcto en las trampas y en las preguntas legítimas que revisé, a costa de respuestas algo más largas. Mi límite: son solo 3 trampas y un modelo con temperatura 0.2, no demuestra que nunca alucine.


In [24]:
# Prompt v2: obliga a comprobar que el dato exacto (modelo, benchmark, métrica) esté en los fragmentos.
PROMPT_V2_SYSTEM = (
    "Answer using ONLY the supplied academic-paper excerpts. "
    "First check whether the excerpts explicitly contain the exact fact the question asks for "
    "(the same model, benchmark and metric). "
    "If they do not, say clearly that the supplied papers do not contain that information. "
    "You may add one short sentence about what related information the excerpts do contain, "
    "but never use outside knowledge, never guess numbers, and never transfer a result from one model "
    "or benchmark to a different one. "
    "Answer in the same language as the question and be concise. "
    "Mention the paper title supporting the answer when there is evidence."
)
prompt_v2 = ChatPromptTemplate.from_messages([
    ("system", PROMPT_V2_SYSTEM),
    ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
])
rag_chain_v2 = ({"context": retriever | format_docs, "question": RunnablePassthrough()}
                | prompt_v2 | llm | StrOutputParser())

import difflib
print("DIFF del prompt de sistema (v1 → v2):")
for line in difflib.unified_diff(PROMPT_V1_SYSTEM.split(". "), PROMPT_V2_SYSTEM.split(". "),
                                 "v1", "v2", lineterm="", n=0):
    print(line)

traps_v2 = run_traps(rag_chain_v2, "prompt v2")

print("\n--- Regresión: ¿v2 sigue respondiendo las preguntas legítimas? ---")
for qid in ("Q1", "Q3", "Q5"):
    a = ask(rag_chain_v2, QUIZ[qid])
    print(f"[v2] {qid} {'✔' if is_hit(qid, a) else '✘'} {a.strip().replace(chr(10), ' ')[:300]}")


DIFF del prompt de sistema (v1 → v2):
--- v1
+++ v2
@@ -1,2 +1,4 @@
-Answer using only the supplied academic-paper excerpts
-If the excerpts do not contain enough evidence, say so explicitly
+Answer using ONLY the supplied academic-paper excerpts
+First check whether the excerpts explicitly contain the exact fact the question asks for (the same model, benchmark and metric)
+If they do not, say clearly that the supplied papers do not contain that information
+You may add one short sentence about what related information the excerpts do contain, but never use outside knowledge, never guess numbers, and never transfer a result from one model or benchmark to a different one
@@ -4 +6 @@
-Mention the paper title supporting the answer.
+Mention the paper title supporting the answer when there is evidence.


  ⏳ error transitorio (Exception); reintento 1 en 3s



[prompt v2] T1: What BLEU score did GPT-4 achieve on WMT 2014 English-to-German?
→ The supplied papers do not contain that information. They only report Transformer results from “Attention Is All You Need,” including 28.4 BLEU on WMT 2014 English-to-German, and no results for GPT-4.



[prompt v2] T2: What exact-match score did RAG obtain on the HotpotQA multi-hop question answering benchmark?
→ The supplied excerpts do not contain RAG’s exact-match score on the HotpotQA multi-hop benchmark; HotpotQA is not mentioned in the provided text. The excerpts do report ExactMatch evaluation for other tasks (e.g., CT and WQ initialized from an NQ RAG model, and TQA), but no HotpotQA result is given.



[prompt v2] T3: How many GPUs and how many training days did Llama 2 70B need, and how does that compare with the base Transformer?
→ The supplied excerpts do not contain any information about Llama 2 70B—no GPU count or training days—so the requested comparison with the base Transformer cannot be made from these papers. The excerpts are from “Attention Is All You Need” and only report Transformer training details, such as the big Transformer on WMT 2014 English-to-German training for 3.5 days on 8 P100 GPUs.

--- Regresión: ¿v2 sigue respondiendo las preguntas legítimas? ---


[v2] Q1 ✔ 28.4 BLEU. This is stated in "Attention Is All You Need," which reports that the Transformer (big) model achieved a new state-of-the-art BLEU score of 28.4 on the WMT 2014 English-to-German translation task.


[v2] Q3 ✔ The supplied papers do contain this information. In "Attention Is All You Need," the scaling factor \(1/\sqrt{d_k}\) is used to counteract the effect that, for large values of \(d_k\), dot products grow large in magnitude, pushing the softmax function into regions where it has extremely small gradie


[v2] Q5 ✔ The base Transformer model uses h = 8 attention heads, with key and value dimensions per head of d_k = d_v = 64 (since d_model = 512 and d_k = d_v = d_model/h). This is stated in “Attention Is All You Need.”


_El análisis de B3 (clasificación y comparación de prompts) está en la celda de arriba y en `RESPUESTAS.md`._

---
## B4 · Preguntas en español sobre papers en inglés

Traduzco Q1, Q3 y Q4. Comparo (a) los chunks recuperados y sus scores, (b) la calidad de la respuesta con los mismos criterios, y (c) hago un mini experimento sobre los embeddings para probar mi hipótesis.

In [25]:
import numpy as np

QUIZ_ES = {
    "Q1": "¿Qué puntaje BLEU obtuvo el Transformer (modelo grande) en la traducción inglés-alemán de WMT 2014?",
    "Q3": "¿Cuál es el propósito del factor de escala sqrt(dk) en la atención de producto punto escalado?",
    "Q4": "¿Cuáles son las dos variantes del modelo RAG propuestas en el artículo y en qué se diferencian?",
}
# Mismos datos clave que en inglés, pero en español
GT_PATTERNS_ES = {
    "Q1": [r"28[\.,]4"],
    "Q3": [r"softmax", r"gradiente|gradient", r"grande|crec|magnitud|alto|elevad"],
    "Q4": [r"RAG-Sequence", r"RAG-Token", r"mismo documento|un solo documento|único documento|mismo texto recuperado|misma",
           r"diferente|distinto|cada token|por token"],
}
def is_hit_es(qid, answer):
    return all(re.search(p, answer, flags=re.I) for p in GT_PATTERNS_ES[qid])

def top_chunks(question, k=5):
    out = retry_call(vectorstore.similarity_search_with_relevance_scores, question, k=k, score_threshold=0.25)
    time.sleep(0.3)
    return [((d.metadata["source"], d.metadata["chunk_index"]), sc) for d, sc in out]

b4 = {}
for qid, q_es in QUIZ_ES.items():
    en, es = top_chunks(QUIZ[qid]), top_chunks(q_es)
    ids_en, ids_es = [c for c, _ in en], [c for c, _ in es]
    inter = set(ids_en) & set(ids_es)
    ans_es = ask(rag_chain, q_es)
    b4[qid] = dict(en=en, es=es, inter=inter, ans_en=ANSWERS_B1[qid], ans_es=ans_es,
                   hit_en=is_hit(qid, ANSWERS_B1[qid]), hit_es=is_hit_es(qid, ans_es))
    print(f"\n=== {qid}\nEN: {QUIZ[qid]}\nES: {q_es}")
    print("  chunks EN:", [(s[:4], i, round(x, 3)) for (s, i), x in en])
    print("  chunks ES:", [(s[:4], i, round(x, 3)) for (s, i), x in es])
    print(f"  chunks en común: {len(inter)} de 5 | top-1 igual: {ids_en[0] == ids_es[0]}")
    print(f"  Respuesta EN ({'✔' if b4[qid]['hit_en'] else '✘'}): {ANSWERS_B1[qid].strip()[:350]}")
    print(f"  Respuesta ES ({'✔' if b4[qid]['hit_es'] else '✘'}): {ans_es.strip()[:350]}")



=== Q1
EN: What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?
ES: ¿Qué puntaje BLEU obtuvo el Transformer (modelo grande) en la traducción inglés-alemán de WMT 2014?
  chunks EN: [('atte', 28, 0.855), ('atte', 1, 0.829), ('atte', 26, 0.79), ('atte', 40, 0.758), ('atte', 31, 0.749)]
  chunks ES: [('atte', 28, 0.854), ('atte', 1, 0.841), ('atte', 26, 0.789), ('atte', 40, 0.773), ('atte', 31, 0.751)]
  chunks en común: 5 de 5 | top-1 igual: True
  Respuesta EN (✔): 28.4 BLEU. (Paper: "Attention Is All You Need")
  Respuesta ES (✔): El Transformer (modelo grande) obtuvo un puntaje BLEU de **28.4** en la tarea de traducción inglés-alemán de WMT 2014, según el paper **“Attention Is All You Need”**.



=== Q3
EN: What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?
ES: ¿Cuál es el propósito del factor de escala sqrt(dk) en la atención de producto punto escalado?
  chunks EN: [('atte', 11, 0.775), ('atte', 12, 0.741), ('atte', 13, 0.69), ('atte', 16, 0.648), ('atte', 20, 0.642)]
  chunks ES: [('atte', 11, 0.673), ('atte', 12, 0.665), ('atte', 13, 0.648), ('atte', 16, 0.598), ('atte', 2, 0.592)]
  chunks en común: 4 de 5 | top-1 igual: True
  Respuesta EN (✔): The scaling factor \(1/\sqrt{d_k}\) counteracts the growth of dot-product magnitudes for large \(d_k\). The paper explains that when query and key components are independent with mean 0 and variance 1, their dot product has mean 0 and variance \(d_k\), so large \(d_k\) pushes the softmax into regions with extremely small gradients. Scaling by \(\sq
  Respuesta ES (✔): En “Attention Is All You Need”, se divide por √dk para evitar que, cuando dk es grande, los productos punto crezcan en magnitud y em


=== Q4
EN: What are the two RAG model variants proposed in the paper and how do they differ?
ES: ¿Cuáles son las dos variantes del modelo RAG propuestas en el artículo y en qué se diferencian?
  chunks EN: [('rag_', 10, 0.755), ('rag_', 11, 0.737), ('rag_', 1, 0.736), ('rag_', 50, 0.732), ('rag_', 7, 0.725)]
  chunks ES: [('rag_', 10, 0.74), ('rag_', 11, 0.721), ('rag_', 50, 0.716), ('rag_', 1, 0.712), ('rag_', 35, 0.71)]
  chunks en común: 4 de 5 | top-1 igual: True
  Respuesta EN (✔): The paper *Retrieval-Augmented Generation* proposes two variants: **RAG-Sequence** and **RAG-Token**. They differ in how they marginalize over retrieved latent documents: **RAG-Sequence** uses the same retrieved document to predict every target token (per-output basis, one document responsible for all tokens), whereas **RAG-Token** can predict each
  Respuesta ES (✔): Las dos variantes son **RAG-Sequence** y **RAG-Token**, según el artículo *“Retrieval-Augmented Generation”* (rag_paper.pdf).

Se difere

In [26]:
# Mini experimento: ¿qué tan cerca quedan en el espacio de embeddings una pregunta y su traducción?
def cos(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

def emb_q(text):
    v = retry_call(embeddings.embed_query, text)
    time.sleep(0.3)
    return v

E = {qid: (emb_q(QUIZ[qid]), emb_q(QUIZ_ES[qid])) for qid in QUIZ_ES}
print("Coseno entre la pregunta EN y su traducción ES:")
for qid, (e_en, e_es) in E.items():
    print(f"  {qid}: {cos(e_en, e_es):.3f}")

# Controles: preguntas EN de OTRO tema, y cada ES contra las EN de las otras preguntas
print("\nControl: coseno entre preguntas EN distintas, y ES contra EN de otro tema:")
ids = list(E)
for i in ids:
    for j in ids:
        if i < j:
            print(f"  EN {i} vs EN {j}: {cos(E[i][0], E[j][0]):.3f} | ES {i} vs EN {j}: {cos(E[i][1], E[j][0]):.3f}")

# ¿Qué tan cerca queda cada versión del chunk que contiene el dato? (Q1: tabla con 28.4)
gold = {"Q1": "28.4", "Q4": "RAG-Sequence"}
all_chunks = vectorstore.similarity_search("x", k=1000)  # todo el índice (150 chunks)
for qid, key in gold.items():
    cand = [d for d in all_chunks if key in d.page_content]
    scored = []
    for d in cand[:6]:
        v = emb_q(d.page_content)
        scored.append((cos(E[qid][0], v), cos(E[qid][1], v), d.metadata["source"][:4], d.metadata["chunk_index"]))
    best = max(scored, key=lambda r: max(r[0], r[1]))
    print(f"\n{qid}: chunk con '{key}' mejor puntuado -> coseno EN={best[0]:.3f}, ES={best[1]:.3f} ({best[2]}#{best[3]})")


Coseno entre la pregunta EN y su traducción ES:
  Q1: 0.920
  Q3: 0.727
  Q4: 0.957

Control: coseno entre preguntas EN distintas, y ES contra EN de otro tema:
  EN Q1 vs EN Q3: 0.118 | ES Q1 vs EN Q3: 0.116
  EN Q1 vs EN Q4: 0.156 | ES Q1 vs EN Q4: 0.134
  EN Q3 vs EN Q4: 0.111 | ES Q3 vs EN Q4: 0.045



Q1: chunk con '28.4' mejor puntuado -> coseno EN=0.708, ES=0.681 (atte#28)



Q4: chunk con 'RAG-Sequence' mejor puntuado -> coseno EN=0.462, ES=0.430 (rag_#11)


## B4 · Preguntas en español sobre papers en inglés

_Evidencia:_ celdas de B4 del notebook (recuperación EN contra ES, respuestas y mini experimento con embeddings reales).

| Pregunta | Chunks en común EN/ES (de 5) | Top-1 igual | Score top-1 EN → ES | Calidad respuesta EN | Calidad respuesta ES |
|---|---|---|---|---|---|
| Q1 BLEU | 5 | sí (chunk 28) | 0.855 → 0.854 | correcta (28.4) | correcta (28.4, en español, cita el paper) |
| Q3 sqrt(dk) | 4 | sí (chunk 11) | 0.775 → 0.673 | correcta | correcta (softmax con gradientes extremadamente pequeños) |
| Q4 variantes RAG | 4 | sí (chunk 10) | 0.755 → 0.740 | correcta | correcta (RAG-Sequence y RAG-Token) |

Mini experimento (coseno entre embeddings de `nemotron-3-embed-1b`): pregunta EN contra su traducción ES: Q1 0.920, Q3 0.727, Q4 0.957. Control, preguntas de temas distintos: entre 0.05 y 0.16. Contra el chunk que contiene el dato: Q1 (chunk con "28.4") EN 0.708 contra ES 0.681; Q4 (chunk con "RAG-Sequence") EN 0.462 contra ES 0.430.

_Mi respuesta / análisis:_ La búsqueda entre idiomas funcionó: el top-1 fue el mismo chunk en las tres preguntas, 13 de 15 chunks coinciden y las tres respuestas en español fueron correctas (con mis patrones en español). Lo que sí cambió fue el nivel de los scores, sobre todo en Q3 (0.775 a 0.673), y la traducción de Q3 es la que queda menos cerca de su original (0.727 contra 0.92 y 0.96 en las otras). Mi hipótesis es que el modelo de embeddings es multilingüe y proyecta frases con el mismo significado a regiones cercanas del mismo espacio, así que la pregunta en español cae cerca de los chunks en inglés; la pérdida es pequeña (unos 0.03 de coseno en el chunk correcto) y es mayor cuando la pregunta depende de símbolos o términos técnicos como sqrt(dk), que no se "traducen". Lo comprobé en parte con el coseno entre pares de preguntas (0.73 a 0.96 contra 0.05 a 0.16 de controles), pero no es una prueba completa: solo son 3 preguntas. Para comprobarla mejor correría las 5 preguntas y más, midiendo hit@k en inglés contra español, y probaría un modelo solo en inglés como contraste.


---
## B5 · Mi propio quiz con *ground truth*

Leí las secciones de los PDFs y escribí tres preguntas nuevas: una del paper de Attention, una del paper de RAG y una que obliga a combinar los dos. Siguen el formato del notebook (código con la pregunta, bloque `<details>` con el *ground truth* y la sección).

In [27]:
# ── Pregunta B5-1 (Attention) ─────────────────────────────────
b5q1 = ("How many identical layers are in the encoder and in the decoder stacks of the base Transformer, "
        "and what is the inner-layer dimensionality of the position-wise feed-forward network?")

b5a1 = ask(rag_chain, b5q1)
print(f"Q: {b5q1}")
print(f"A: {b5a1}")


  ⏳ error transitorio (Exception); reintento 1 en 2s


Q: How many identical layers are in the encoder and in the decoder stacks of the base Transformer, and what is the inner-layer dimensionality of the position-wise feed-forward network?
A: The base Transformer has **N = 6 identical layers** in both the encoder stack and the decoder stack, and the position-wise feed-forward network has an inner-layer dimensionality of **d_ff = 2048** (with input/output dimensionality d_model = 512).

Source: *Attention Is All You Need* (Sections 3.1 and 3.3).


<details>
<summary>✅ Ground Truth — B5-1</summary>

Tanto el encoder como el decoder están formados por una pila de **N = 6** capas idénticas.
La red feed-forward de cada capa tiene dimensionalidad de entrada y salida d_model = 512 y una capa interna de **d_ff = 2048**.
*(Fuente: Sections 3.1 y 3.3 — Attention Is All You Need)*
</details>

In [28]:
# ── Pregunta B5-2 (RAG) ───────────────────────────────────────
b5q2 = ("Which Wikipedia dump does RAG use as its non-parametric knowledge source, how are the articles split, "
        "and how many documents does the index contain?")

b5a2 = ask(rag_chain, b5q2)
print(f"Q: {b5q2}")
print(f"A: {b5a2}")


Q: Which Wikipedia dump does RAG use as its non-parametric knowledge source, how are the articles split, and how many documents does the index contain?
A: RAG uses the December 2018 Wikipedia dump as its non-parametric knowledge source. Each Wikipedia article is split into disjoint 100-word chunks, producing a total of 21M documents in the index. (Source: "Retrieval-Augmented Generation" | rag_paper.pdf)


<details>
<summary>✅ Ground Truth — B5-2</summary>

RAG usa el volcado de Wikipedia de **diciembre de 2018**. Cada artículo se divide en fragmentos disjuntos de **100 palabras**,
para un total de **21 millones de documentos**. Con el codificador de documentos calculan un embedding por documento y construyen
un índice MIPS con FAISS y una aproximación Hierarchical Navigable Small World.
*(Fuente: Section 3 "Experiments", párrafo inicial — RAG paper)*
</details>

In [29]:
# ── Pregunta B5-3 (combina los dos papers) ───────────────────
b5q3 = ("Both papers train their models with the Adam optimizer. Which beta values and learning-rate warmup "
        "does the Transformer paper use, and which component of RAG is kept fixed (not fine-tuned) during training?")

b5a3 = ask(rag_chain, b5q3)
print(f"Q: {b5q3}")
print(f"A: {b5a3}")


Q: Both papers train their models with the Adam optimizer. Which beta values and learning-rate warmup does the Transformer paper use, and which component of RAG is kept fixed (not fine-tuned) during training?
A: The Transformer paper (“Attention Is All You Need”) uses Adam with β₁ = 0.9, β₂ = 0.98, ε = 10⁻⁹, and a learning-rate schedule with warmup_steps = 4000 (linear increase for the first 4,000 steps, then inverse-square-root decay).

The RAG paper (“Retrieval-Augmented Generation”) keeps the document encoder (and its index) fixed during training, fine-tuning only the query encoder BERT and the BART generator.


<details>
<summary>✅ Ground Truth — B5-3 (combina los dos papers)</summary>

**Attention Is All You Need (Sección 5.3):** Adam con β1 = 0.9, β2 = 0.98 y ε = 10^-9; la tasa de aprendizaje crece linealmente durante
**warmup_steps = 4000** pasos y luego decrece proporcionalmente a la raíz cuadrada inversa del paso.
**RAG:** también entrena con Adam, pero mantiene **fijos el codificador de documentos (BERT_d) y, con él, el índice de documentos**;
solo ajusta el codificador de consultas (BERT_q) y el generador BART, porque actualizar el índice sería costoso.
*(Fuentes: Section 5.3 del Transformer y Section 3 — Experiments del RAG paper)*
</details>

In [30]:
B5 = {
    "B5-1": (b5q1, b5a1, [r"\b6\b|six", r"2048"], "attention_paper.pdf"),
    "B5-2": (b5q2, b5a2, [r"December 2018|2018", r"100.word", r"21 ?M|21 million"], "rag_paper.pdf"),
    "B5-3": (b5q3, b5a3, [r"0\.98", r"4000|4,000", r"document encoder|BERT.?d|index"], "ambos"),
}
for bid, (question, ans, pats, exp) in B5.items():
    scored = retry_call(vectorstore.similarity_search_with_relevance_scores, question, k=5, score_threshold=0.25)
    time.sleep(0.3)
    n_att = sum(d.metadata["source"] == "attention_paper.pdf" for d, _ in scored)
    n_rag = sum(d.metadata["source"] == "rag_paper.pdf" for d, _ in scored)
    hit = all(re.search(p, ans, flags=re.I) for p in pats)
    print(f"\n=== {bid} · paper esperado: {exp}")
    for d, sc in scored:
        print(f"  {sc:.3f} | {d.metadata['source']:<20} | chunk {d.metadata['chunk_index']:>3} | "
              f"{d.page_content[:70].replace(chr(10), ' ')}")
    print(f"  chunks: attention={n_att}, rag={n_rag} | datos clave completos en la respuesta: {hit}")



=== B5-1 · paper esperado: attention_paper.pdf
  0.823 | attention_paper.pdf  | chunk   9 | Figure1: TheTransformer-modelarchitecture. TheTransformerfollowsthisov
  0.812 | attention_paper.pdf  | chunk  16 | allpositionsinthedecoderuptoandincludingthatposition. Weneedtopreventl
  0.770 | attention_paper.pdf  | chunk  15 | 3.2.3 ApplicationsofAttentioninourModel TheTransformerusesmulti-headat
  0.762 | attention_paper.pdf  | chunk  10 | layers,produceoutputsofdimensiond =512. model Decoder: Thedecoderisals
  0.744 | attention_paper.pdf  | chunk  17 | The dimensionality of input and output is d = 512, and the inner-layer
  chunks: attention=5, rag=0 | datos clave completos en la respuesta: True



=== B5-2 · paper esperado: rag_paper.pdf
  0.752 | rag_paper.pdf        | chunk  18 | wecanmakeafurtherapproximationthatp (y|x,z )≈0whereywasnotgenerateddur
  0.751 | rag_paper.pdf        | chunk   1 | trainedmodelswithadifferentiableaccessmechanismtoexplicitnon-parametri
  0.734 | rag_paper.pdf        | chunk   7 | Weendowpre-trained,parametric-memorygenerationmodelswithanon-parametri
  0.731 | rag_paper.pdf        | chunk  42 | |                     |       |      |            |           |       
  0.724 | rag_paper.pdf        | chunk  37 | p(z|x).Table6showstheresults.ForFEVER,BM25performsbest,perhapssinceFEV
  chunks: attention=0, rag=5 | datos clave completos en la respuesta: True



=== B5-3 · paper esperado: ambos
  0.758 | rag_paper.pdf        | chunk  16 | (cid:80) minimizethenegativemarginallog-likelihoodofeachtarget, −logp(
  0.728 | rag_paper.pdf        | chunk  10 | 2 Methods WeexploreRAGmodels,whichusetheinputsequencextoretrievetextdo
  0.720 | attention_paper.pdf  | chunk  25 | trainedthebasemodelsforatotalof100,000stepsor12hours. Forourbigmodels,
  0.714 | rag_paper.pdf        | chunk  91 | (iii)aneuralnetworktopredictthelogit. Wedidnotfindthattheseimprovedper
  0.710 | rag_paper.pdf        | chunk   7 | Weendowpre-trained,parametric-memorygenerationmodelswithanon-parametri
  chunks: attention=1, rag=4 | datos clave completos en la respuesta: True


## B5 · Escribe tu propio quiz con ground truth

_Evidencia:_ celdas B5-1, B5-2 y B5-3 añadidas al notebook, cada una con su bloque `<details>` de ground truth, y sus fuentes en los PDFs.

| # | Pregunta | Ground truth y sección | Respuesta del RAG | Datos clave completos | Papers recuperados |
|---|---|---|---|---|---|
| B5-1 | Capas idénticas de encoder y decoder del Transformer base y dimensión interna de la feed-forward | N = 6 en ambos, d_ff = 2048 (d_model = 512). Secciones 3.1 y 3.3, Attention | N = 6 en ambos stacks y d_ff = 2048, cita secciones 3.1 y 3.3 | sí | Attention 5, RAG 0 |
| B5-2 | Volcado de Wikipedia, cómo se divide y cuántos documentos tiene el índice de RAG | Diciembre 2018, fragmentos disjuntos de 100 palabras, 21M documentos (índice MIPS con FAISS y HNSW). Sección 3, Experiments, RAG | Volcado de diciembre de 2018, chunks de 100 palabras, 21M documentos | sí | Attention 0, RAG 5 |
| B5-3 (combina los dos) | Betas y warmup de Adam en el Transformer y qué componente de RAG queda fijo | Transformer: β1 = 0.9, β2 = 0.98, ε = 1e-9, warmup 4000 (Sección 5.3). RAG: el codificador de documentos (BERT_d) y su índice quedan fijos (Sección 3, Experiments) | Dio β 0.9 y 0.98, ε 1e-9, warmup 4000, y que RAG mantiene fijo el codificador de documentos y su índice, ajustando solo el de consultas y BART | sí | Attention 1, RAG 4 |

_Mi respuesta / análisis:_ Las tres preguntas salieron bien, y la combinada fue la más interesante: recuperó 4 chunks de RAG y solo 1 de Attention (el chunk 25 de entrenamiento), y aun así la respuesta incluyó los hiperparámetros de Adam del Transformer, así que ese único chunk bastó. En B5-2 los 5 chunks mostrados tienen solo el inicio visible (70 caracteres), por lo que no puedo decir cuál de ellos contiene el "21M"; lo que sí sé es que la respuesta coincide con el ground truth que verifiqué leyendo el PDF. Mi límite: esperaba que la pregunta de dos papers fuera la que más fallara y no fue así, pero con 3 preguntas no hay base para generalizar.


---
## Reflexión final

1. **¿Qué ocurrió cuando el contexto no contenía información suficiente?** ¿El modelo reconoció la limitación o inventó una respuesta?
2. **¿Qué fragmentos recuperó para la pregunta bonus?** ¿Aparecieron ambos papers?
3. **¿Qué efecto tendría modificar `chunk_size`, `chunk_overlap`, `k` o `score_threshold`?**
4. **¿Por qué conviene separar el modelo de embeddings del modelo generativo?**
5. **¿Qué cambiarías para llevar este prototipo a producción?** Considera persistencia, control de acceso, evaluación, observabilidad y costos.


## Reflexión

1. **¿Qué ocurrió cuando el contexto no contenía información suficiente?** En B3 las tres preguntas trampa (GPT-4 en WMT14, HotpotQA, Llama 2 70B) recibieron cinco chunks con scores de 0.66 a 0.77 y, aun así, el modelo reconoció la limitación en las tres en lugar de inventar. En T3 además separó lo que sí sabía (3.5 días en 8 P100 del Transformer) de lo que no (Llama 2). Eso se debe en buena parte a la instrucción explícita del prompt; probé un prompt más estricto (v2) y mantuvo el mismo comportamiento. Con el umbral 0.80 de B2, cuando el retriever devolvió 0 chunks en cuatro preguntas, el modelo también dijo que no tenía evidencia, y por eso esa configuración sacó 1/5.

2. **¿Qué fragmentos recuperó para la pregunta bonus? ¿Aparecieron ambos papers?** Recuperó 5 chunks: 4 del paper de RAG (chunks 7, 11, 10, 12 con scores 0.765 a 0.737) y 1 de Attention (el chunk 0, score 0.737, que es la portada). Sí aparecieron ambos papers, pero la parte de Attention fue marginal: la respuesta correcta (BART, basado en el Transformer) salió del texto de RAG. Un chunk de la sección de arquitectura de Attention habría sido mejor evidencia, y eso muestra que una pregunta que cruza documentos no garantiza una recuperación equilibrada.

3. **¿Qué efecto tuvo modificar chunk_size, chunk_overlap, k o score_threshold?** El tamaño del chunk cambió el índice de 326 chunks (400) a 150 (900) y 96 (1500), y fue lo único que movió los aciertos (4/5, 5/5 y 4/5, con la salvedad de que uno de esos fallos fue un falso negativo de mi criterio). Con k = 3, 5 y 8 el resultado fue 5/5 en los tres y solo creció el contexto enviado al modelo. El score_threshold de 0.15, 0.25 y 0.40 no tuvo efecto porque los scores de este modelo van de 0.63 a 0.86 en la escala (coseno+1)/2; solo con 0.80 empezó a filtrar. El overlap lo ajusté de forma proporcional (13 %) y no lo aislé como variable.

4. **¿Por qué conviene separar el modelo de embeddings del modelo generativo?** Porque hacen trabajos distintos y se cambian a ritmos distintos: el de embeddings (`nemotron-3-embed-1b`, 1B parámetros, dimensión 2048) se usa una vez por chunk al indexar y una vez por consulta, mientras que el generativo es de 120B y se llama por respuesta. Hoy lo viví: mi cuenta no tenía habilitado el generativo original (404) y pude cambiarlo por `nemotron-3.5-super-vl-120b-a12b` sin tocar el índice ni reconstruir embeddings. Si fueran el mismo modelo, cambiar el generador obligaría a reindexar, y el costo del experimento de chunk_size (3 construcciones del índice) muestra que reindexar no es gratis.

5. **¿Qué cambiarías para llevar este prototipo a producción?** Persistencia: hoy Qdrant corre en memoria y cada reinicio paga embeddings otra vez (indexar tardó 4 a 12 s aquí, pero con miles de documentos serían minutos y dinero), así que usaría Qdrant con disco o servidor. Control de acceso: filtros por metadata y permisos por documento, porque ahora cualquiera consulta todo. Evaluación: automatizar lo que hice a mano con un conjunto fijo de preguntas, hit@k y pruebas de trampas en cada cambio, y repetir cada configuración varias veces porque el modelo tiene ruido. Observabilidad y costos: registrar consultas, chunks, scores y latencia, y vigilar los límites de la API (hoy vi errores 429 y timeouts que tuve que reintentar con backoff). También corregiría la extracción de PDF, ya que en el paper de Attention salen palabras pegadas, y reemplazaría el umbral por uno calibrado a la escala real de scores.
